# SFT Fine-Tuning: LLaVA-NeXT 7B on Place Pulse 2.0

**Model:** `llava-hf/llava-v1.6-mistral-7b-hf`  
**Method:** Supervised Fine-Tuning (SFT) with QLoRA + PyTorch Lightning  
**Dataset:** Place Pulse 2.0 — 6 urban perceptual attributes  
**Hardware:** A100 80GB (enable High RAM slider)  
**Based on:** Farzad-R/Finetune-LLAVA-NEXT (same model, same training approach)

---
### Key Design Decisions
- **PyTorch Lightning** instead of SFTTrainer — proven to work with LLaVA-NeXT
- **Position randomization** — winner placed randomly left/right during training to prevent positional bias
- **Vote aggregation + majority ratio >= 0.7** — only high-confidence pairs used for training
- **CoT evaluation only** — with position swap to eliminate positional bias, matching original notebook

---
### Baseline CoT results to beat (LLaVA-1.5-7B, no fine-tuning):
| Attribute | CoT Acc | CoT Tie Rate |
|---|---|---|
| Beautiful | 0.590 | 0.51 |
| Safer | 0.540 | 0.42 |
| Livelier | 0.590 | 0.69 |
| More Boring | 0.510 | 0.61 |
| Wealthier | 0.540 | 0.46 |
| More Depressing | 0.500 | 0.40 |

---
## Section 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    accelerate>=0.28.0 \
    bitsandbytes>=0.43.0 \
    lightning>=2.0.0 \
    Pillow \
    tqdm \
    pandas

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU available: True
GPU name: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.1


---
## Section 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/llava_next_sft"
FINAL_MODEL_DIR  = os.path.join(DRIVE_OUTPUT_DIR, "final_model")
RESULTS_DIR      = os.path.join(DRIVE_OUTPUT_DIR, "results")

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Drive directories ready.")

Mounted at /content/drive
Drive directories ready.


---
## Section 3: Imports and Configuration

In [ ]:
import torch
import pandas as pd
import os
import re
import random
from PIL import Image as PILImage
from tqdm import tqdm

# ============================================================
# Global configuration
# Adjust SAMPLES_PER_ATTR based on compute budget:
# 100/attr  = ~540 training pairs  = ~15 min  (quick test)
# 500/attr  = ~2700 training pairs = ~1.5 hrs (good results)
# 2000/attr = ~10800 training pairs= ~6 hrs   (best results)
# ============================================================
MODEL_ID         = "llava-hf/llava-v1.6-mistral-7b-hf"
MAX_LENGTH       = 256
SAMPLES_PER_ATTR = 100

---
## Section 4: Load Raw Data

In [ ]:
# Only unzip if not already done
if not os.path.exists("/content/votes.tsv"):
    !unzip -q "/content/drive/MyDrive/place-pulse-2.0 (2).zip" -d /content/
    print("Unzip complete.")
else:
    print("Already unzipped, skipping.")

Unzip complete.


In [ ]:
df_votes   = pd.read_csv("/content/votes.tsv", sep="\t")
df_studies = pd.read_csv("/content/studies.tsv", sep="\t")
image_dir  = "/content/images"

print("Votes shape:  ", df_votes.shape)
print("Studies shape:", df_studies.shape)
print("Total images: ", len(os.listdir(image_dir)))

Votes shape:   (1555561, 7)
Studies shape: (6, 6)
Total images:  110988


---
## Section 5: Data Cleaning

Follows original notebook exactly:
- Filter valid choices (left/right only)
- Build pair-level dataset with vote counts
- Remove tie pairs
- Apply majority ratio >= 0.7 filter (only high-confidence pairs)

In [ ]:
# Step 1: Keep only valid choices — matches original Cell 7
df_votes = df_votes[df_votes["choice"].isin(["left", "right"])].copy()
print("Valid votes:", len(df_votes))

Valid votes: 1350546


In [ ]:
# Step 2: Build image map — matches original Cell 17
image_map = {}
for file in os.listdir(image_dir):
    if file.lower().endswith(".jpg"):
        parts = file.split("_")
        if len(parts) >= 3:
            image_id = parts[2]  # correct ID position
            image_map[image_id] = os.path.join(image_dir, file)
print("Total images mapped:", len(image_map))

Total images mapped: 110988


In [ ]:
# Step 3: Attribute mapping
ATTRIBUTE_MAP = {
    "beautiful":       "5217c351ad93a7d3e7b07a64",
    "safer":           "50a68a51fdc9f05596000002",
    "livelier":        "50f62c41a84ea7c5fdd2e454",
    "more boring":     "50f62c68a84ea7c5fdd2e456",
    "wealthier":       "50f62cb7a84ea7c5fdd2e458",
    "more depressing": "50f62ccfa84ea7c5fdd2e459",
}
STUDY_TO_ATTR = {v: k for k, v in ATTRIBUTE_MAP.items()}

df_votes["attribute"] = df_votes["study_id"].map(STUDY_TO_ATTR)
df_votes = df_votes.dropna(subset=["attribute"]).copy()
print("Votes per attribute:")
print(df_votes["attribute"].value_counts())

Votes per attribute:
attribute
safer              443052
livelier           320803
beautiful          190606
wealthier          150602
more depressing    125030
more boring        120453
Name: count, dtype: int64


In [ ]:
# Step 4: Build pair-level dataset — matches original Cells 8-11

# Create pair identifier
df_votes["pair_id"] = df_votes.apply(
    lambda x: "_".join(
        sorted([str(x["left"]), str(x["right"])])
    ) + "_" + str(x["study_id"]),
    axis=1
)

# Count left wins
left_votes = (
    df_votes[df_votes["choice"].isin(["left", "A", 0])]
    .groupby(["pair_id", "left", "right", "study_id"])
    .size()
    .reset_index(name="left_votes")
)

# Count right wins
right_votes = (
    df_votes[df_votes["choice"].isin(["right", "B", 1])]
    .groupby(["pair_id", "left", "right", "study_id"])
    .size()
    .reset_index(name="right_votes")
)

# Merge vote counts
pair_df = pd.merge(
    left_votes, right_votes,
    on=["pair_id", "left", "right", "study_id"],
    how="outer"
)
pair_df["left_votes"]  = pair_df["left_votes"].fillna(0)
pair_df["right_votes"] = pair_df["right_votes"].fillna(0)
pair_df["total_votes"] = pair_df["left_votes"] + pair_df["right_votes"]

print("Pairs before filtering:", len(pair_df))

# Determine winner
pair_df["winner"] = pair_df.apply(
    lambda x: "left"  if x["left_votes"]  > x["right_votes"]
    else ("right" if x["right_votes"] > x["left_votes"] else "tie"),
    axis=1
)

# Remove tie pairs
pair_df = pair_df[pair_df["winner"] != "tie"].copy()
print("Pairs after removing ties:", len(pair_df))

# Majority ratio filter — keep only confident pairs (>= 0.7)
pair_df["majority_ratio"] = pair_df.apply(
    lambda x: max(x["left_votes"], x["right_votes"]) / x["total_votes"],
    axis=1
)
pair_df = pair_df[pair_df["majority_ratio"] >= 0.7].copy()
print("Final clean pairs after majority ratio filter:", len(pair_df))

# Add image paths
pair_df["img_A"] = pair_df["left"].map(image_map)
pair_df["img_B"] = pair_df["right"].map(image_map)
pair_df = pair_df.dropna(subset=["img_A", "img_B"]).copy()

# Add attribute column
pair_df["attribute"] = pair_df["study_id"].map(STUDY_TO_ATTR)
pair_df = pair_df.dropna(subset=["attribute"]).copy()

print("\nFinal pairs per attribute:")
print(pair_df["attribute"].value_counts())

Pairs before filtering: 1333630
Pairs after removing ties: 1333236
Final clean pairs after majority ratio filter: 1332944

Final pairs per attribute:
attribute
safer              434608
livelier           314355
beautiful          187838
wealthier          147972
more depressing    122559
more boring        118129
Name: count, dtype: int64


---
## Section 6: Attribute Phrases and Visual Cues

In [ ]:
# Attribute phrases — for prompts
ATTRIBUTE_PHRASES = {
    "beautiful":       "beautiful",
    "safer":           "safe",
    "livelier":        "lively",
    "more boring":     "boring",
    "wealthier":       "wealthy",
    "more depressing": "depressing",
}

# Visual cues — matches original notebook Cell 22 exactly
ATTRIBUTE_CUES = {
    "beautiful": [
        "architecture",
        "greenery",
        "cleanliness",
        "visual harmony",
        "scenic features"
    ],
    "safer": [
        "well-lit areas",
        "visible people nearby",
        "clean and maintained streets",
        "no signs of damage or neglect",
        "active businesses or shops"
    ],
    "wealthier": [
        "modern buildings",
        "expensive cars",
        "clean streets",
        "landscaping",
        "well maintained infrastructure"
    ],
    "livelier": [
        "many people visible",
        "movement or activity",
        "vehicles or pedestrians",
        "open shops or markets",
        "bright lights or signs"
    ],
    "more boring": [
        "plain buildings",
        "empty streets",
        "lack of activity",
        "few people",
        "dull environment"
    ],
    "more depressing": [
        "abandoned buildings",
        "poor lighting",
        "empty streets",
        "graffiti",
        "neglected infrastructure"
    ]
}

---
## Section 7: Dataset Preparation

**SFT-specific design decisions:**
- Both images concatenated side by side into ONE combined image
- Winner position randomized 50/50 (left=Image A, right=Image B) to prevent positional bias
- Dataset returns `(combined_image, question, answer)` tuples — following Farzad-R pattern
- Training uses only high-confidence pairs (majority_ratio >= 0.7)

In [ ]:
# Helper: concatenate two PIL images side by side
# Used for both training (in dataset) and evaluation
def concat_pil_images(img_left, img_right, max_size=672):
    img_left  = img_left.convert("RGB")
    img_right = img_right.convert("RGB")
    img_left.thumbnail((max_size, max_size))
    img_right.thumbnail((max_size, max_size))
    h         = min(img_left.height, img_right.height)
    img_left  = img_left.resize((int(img_left.width  * h / img_left.height),  h))
    img_right = img_right.resize((int(img_right.width * h / img_right.height), h))
    combined  = PILImage.new("RGB", (img_left.width + img_right.width, h))
    combined.paste(img_left,  (0, 0))
    combined.paste(img_right, (img_left.width, 0))
    return combined


# Helper: concatenate from file paths (used in evaluation)
def concatenate_images(path_A, path_B, max_size=672):
    img_A = PILImage.open(path_A).convert("RGB")
    img_B = PILImage.open(path_B).convert("RGB")
    return concat_pil_images(img_A, img_B, max_size)

In [ ]:
# Sample training pairs per attribute
random.seed(42)
sampled_dfs = []
for attr, group in pair_df.groupby("attribute"):
    n = min(SAMPLES_PER_ATTR, len(group))
    sampled_dfs.append(group.sample(n=n, random_state=42))
    print(f"{attr}: {n} pairs sampled")

train_df = pd.concat(sampled_dfs).reset_index(drop=True)
print(f"\nTotal training pairs: {len(train_df)}")

beautiful: 100 pairs sampled
livelier: 100 pairs sampled
more boring: 100 pairs sampled
more depressing: 100 pairs sampled
safer: 100 pairs sampled
wealthier: 100 pairs sampled

Total training pairs: 600


In [ ]:
from torch.utils.data import Dataset

class PlacePulseDataset(Dataset):
    """
    PyTorch Dataset for Place Pulse SFT.
    Follows Farzad-R's LlavaDataset pattern.

    Returns (combined_image, question, answer) tuples where:
    - combined_image: PIL image with both scenes side by side
    - question: the prompt text (without answer)
    - answer: 'Image A' or 'Image B' — matched to winner position

    KEY: winner position is randomized 50/50 to prevent positional bias.
    The model MUST look at the image to answer correctly.
    """

    def __init__(self, dataframe, seed=42):
        self.df = dataframe.reset_index(drop=True)
        # Pre-compute random positions for reproducibility
        random.seed(seed)
        self.winner_on_left = [random.random() < 0.5 for _ in range(len(self.df))]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        attr     = row["attribute"]
        phrase   = ATTRIBUTE_PHRASES[attr]
        cues     = ATTRIBUTE_CUES.get(attr, [])
        cue_text = ", ".join(cues)

        # Load images
        img_A = PILImage.open(row["img_A"]).convert("RGB")
        img_B = PILImage.open(row["img_B"]).convert("RGB")

        # Identify winner and loser from cleaned pair_df
        winner_img = img_A if row["winner"] == "left" else img_B
        loser_img  = img_B if row["winner"] == "left" else img_A

        # Randomly place winner on left (Image A) or right (Image B)
        # This is the key step that prevents positional bias
        if self.winner_on_left[idx]:
            combined = concat_pil_images(winner_img, loser_img)
            answer   = "Image A"
        else:
            combined = concat_pil_images(loser_img, winner_img)
            answer   = "Image B"

        # Question text only — collator adds the full prompt + answer
        question = (
            f"Two places are shown side-by-side:\n\n"
            f"Image A (left)\n"
            f"Image B (right)\n\n"
            f"Your task is to determine which place looks more {attr}.\n\n"
            f"Important visual cues related to \"{attr}\" may include:\n"
            f"{cue_text}\n\n"
            f"Which place looks more {phrase}?\n\n"
            f"Answer ONLY with: Image A or Image B"
        )

        return combined, question, answer


# 90/10 train/val split
split_idx  = int(0.9 * len(train_df))
train_data = PlacePulseDataset(train_df.iloc[:split_idx], seed=42)
val_data   = PlacePulseDataset(train_df.iloc[split_idx:], seed=99)

print(f"Train size: {len(train_data)}")
print(f"Val size:   {len(val_data)}")

# Quick sanity check
img, q, ans = train_data[0]
print(f"\nSample answer: {ans}")
print(f"Image A answers: {sum(1 for i in range(len(train_data)) if train_data.winner_on_left[i])}")
print(f"Image B answers: {sum(1 for i in range(len(train_data)) if not train_data.winner_on_left[i])}")
print("Balance looks good if both counts are roughly equal!")

Train size: 540
Val size:   60

Sample answer: Image B
Image A answers: 269
Image B answers: 271
Balance looks good if both counts are roughly equal!


---
## Section 8: Load Model and Processor

Following Farzad-R exactly:
- `padding_side = "right"` during training
- `bnb_4bit_compute_dtype = torch.float16`
- `find_all_linear_names()` for dynamic LoRA target detection
- `prepare_model_for_kbit_training` + `get_peft_model` explicitly called

In [ ]:
from transformers import AutoProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Load processor — RIGHT padding during training (Farzad-R)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

# QLoRA config — float16 same as Farzad-R
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load model
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The image processor of type `LlavaNextImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# LoRA setup — following Farzad-R exactly
def find_all_linear_names(model):
    """Find all linear layer names excluding multimodal projector and vision model."""
    cls = torch.nn.Linear
    lora_module_names = set()
    multimodal_keywords = ['multi_modal_projector', 'vision_model']
    for name, module in model.named_modules():
        if any(mm_keyword in name for mm_keyword in multimodal_keywords):
            continue
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    return list(lora_module_names)


lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=find_all_linear_names(model),
    init_lora_weights="gaussian",
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 22,151,168 || all params: 7,588,898,816 || trainable%: 0.2919


---
## Section 9: Collators

Following Farzad-R's collator pattern:
- `train_collate_fn`: builds full prompt+answer string, masks padding tokens with -100
- `eval_collate_fn`: prompt only (no answer) for generation during validation
- Images passed directly to `processor(text=texts, images=images, ...)` — no intermediate format

**Training prompt format** (backslash before INST — Farzad-R training style):
`[INST] <image>\n{question} [\INST] {answer}`

**Inference prompt format** (forward slash — Farzad-R test style):
`[INST] <image>\n{question} [/INST]`

In [ ]:
def train_collate_fn(examples):
    images = []
    texts  = []

    for combined_img, question, answer in examples:
        images.append(combined_img)
        prompt = f"[INST] <image>\n{question} [\\INST] {answer}"
        texts.append(prompt)

    batch = processor(
        text=texts,
        images=images,
        padding=True,
        truncation=False,    # ← remove truncation entirely
        return_tensors="pt"
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels

    return (
        batch["input_ids"],
        batch["attention_mask"],
        batch["pixel_values"],
        batch["image_sizes"],
        batch["labels"]
    )


def eval_collate_fn(examples):
    """
    Eval collator — prompt only, no answer.
    Forward slash at inference (Farzad-R test notebook style).
    """
    images  = []
    texts   = []
    answers = []

    for combined_img, question, answer in examples:
        images.append(combined_img)
        # Inference prompt — forward slash, no answer
        prompt = f"[INST] <image>\n{question} [/INST]"
        texts.append(prompt)
        answers.append(answer)

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    return (
        batch["input_ids"],
        batch["attention_mask"],
        batch["pixel_values"],
        batch["image_sizes"],
        answers
    )


print("Collators defined.")

Collators defined.


---
## Section 10: PyTorch Lightning Training Module

Adapted from Farzad-R's `LlavaModelPLModule`.

In [ ]:
import lightning as L
from torch.utils.data import DataLoader


class LlavaPlacePulsePLModule(L.LightningModule):
    """
    PyTorch Lightning module for SFT of LLaVA-NeXT on Place Pulse.
    Adapted from Farzad-R's LlavaModelPLModule.
    """

    def __init__(self, config, processor, model):
        super().__init__()
        self.config     = config
        self.processor  = processor
        self.model      = model
        self.batch_size = config.get("batch_size")

    def training_step(self, batch, batch_idx):
        input_ids, attention_mask, pixel_values, image_sizes, labels = batch

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_sizes=image_sizes,
            labels=labels
        )
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_ids, attention_mask, pixel_values, image_sizes, answers = batch

        # Generate predictions
        generated_ids = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_sizes=image_sizes,
            max_new_tokens=10,
            do_sample=False,
        )
        # Decode only new tokens — chop off the prompt
        predictions = self.processor.batch_decode(
            generated_ids[:, input_ids.size(1):],
            skip_special_tokens=True
        )

        # Compute accuracy
        correct = 0
        for pred, answer in zip(predictions, answers):
            pred   = pred.strip().lower()
            answer = answer.strip().lower()
            if answer in pred or pred in answer:
                correct += 1
            if self.config.get("verbose", False):
                print(f"Pred: '{pred}' | GT: '{answer}' | Correct: {answer in pred}")

        acc = correct / len(answers) if answers else 0
        self.log("val_acc", acc, prog_bar=True)
        return acc

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.config.get("lr")
        )

    def train_dataloader(self):
        return DataLoader(
            train_data,
            collate_fn=train_collate_fn,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=4
        )

    def val_dataloader(self):
        return DataLoader(
            val_data,
            collate_fn=eval_collate_fn,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=4
        )


print("LlavaPlacePulsePLModule defined.")

LlavaPlacePulsePLModule defined.


---
## Section 11: Train

In [ ]:
# Training config — following Farzad-R:
# batch_size=1, accumulate_grad_batches=8, lr=1e-4, precision=16-mixed
config = {
    "max_epochs":              1,
    "check_val_every_n_epoch": 1,
    "gradient_clip_val":       1.0,
    "accumulate_grad_batches": 8,
    "lr":                      1e-4,
    "batch_size":              1,
    "verbose":                 False,
}

model_module = LlavaPlacePulsePLModule(config, processor, model)
print("Module ready.")

Module ready.


In [ ]:
%%time

trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=config.get("max_epochs"),
    accumulate_grad_batches=config.get("accumulate_grad_batches"),
    check_val_every_n_epoch=config.get("check_val_every_n_epoch"),
    gradient_clip_val=config.get("gradient_clip_val"),
    precision="16-mixed",
    limit_val_batches=10,
    num_sanity_val_steps=0,
    log_every_n_steps=10,
)

trainer.fit(model_module)
print("Training complete!")

INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.

┏━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ PeftModel │  3.9 B │ train │     0 │
└───┴───────┴───────────┴────────┴───────┴───────┘

Trainable params: 22.2 M                                                                                           
Non-trainable params: 3.9 B                                                                                        
Total params: 3.9 B                                                                                                
Total estimated model params size (MB): 15.8 K                                                                     
Modules in train mode: 2962                                                                                        
Modules in eval mode: 725                                                                                          
Total FLOPs: 0

Output()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 1. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
INFO: `Trainer.fit` stopped: `max_epochs=1` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


Training complete!
CPU times: user 7min 29s, sys: 54.4 s, total: 8min 23s
Wall time: 8min 28s


---
## Section 12: Save Model

In [ ]:
print(f"Saving model to: {FINAL_MODEL_DIR}")
model_module.model.save_pretrained(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
print("Model saved successfully!")

Saving model to: /content/drive/MyDrive/llava_next_sft/final_model
Model saved successfully!


---
## Section 13: Evaluation — CoT Pairwise Only

**Evaluation follows original notebook exactly:**
- Samples from `df_votes` directly (individual votes as ground truth)
- Filters to pairs where both images exist in `image_map`
- Runs inference TWICE — normal order then swapped — to eliminate positional bias
- Flips second prediction after swapping
- Tie detected when both predictions disagree after swap
- CoT reasoning prompt with step-by-step format
- Parser checks `assistant:` split first, then `answer: a/b`, then `image a/b`

**Skip the model reload cell if evaluating in the same session as training.**

In [ ]:
# ============================================================
# SKIP THIS CELL if evaluating in the same session as training
# Only run if starting a NEW Colab session after training
# ============================================================

from transformers import AutoProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import torch

MODEL_ID   = "llava-hf/llava-v1.6-mistral-7b-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
# Switch to LEFT padding for inference
processor = AutoProcessor.from_pretrained(FINAL_MODEL_DIR)
processor.tokenizer.padding_side = "left"

base_model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_DIR)
model.eval()
print("Fine-tuned SFT model loaded.")

In [ ]:
# Switch to LEFT padding for inference (same session)
processor.tokenizer.padding_side = "left"
model_module.model.eval()

# Use model_module.model for inference in same session
# Use 'model' if you loaded from Drive in the cell above
inference_model = model_module.model
print("Ready for evaluation.")

Ready for evaluation.


In [ ]:
# Parser — adapted from original notebook parse_prediction
# Checks assistant: split first, then answer: a/b, then image a/b
# Adapted for LLaVA-NeXT [/INST] format
def parse_prediction(text):
    if text is None:
        return None

    text = text.lower().strip()

    # Strip prompt — check both formats
    if "[/inst]" in text:
        text = text.split("[/inst]")[-1].strip()
    if "assistant:" in text:
        text = text.split("assistant:")[-1].strip()

    # CoT format — answer: a or answer: b
    if "answer: a" in text: return 1
    if "answer: b" in text: return 0

    # Direct format — image a or image b
    if "image a" in text: return 1
    if "image b" in text: return 0

    # Fallback
    if text == "a" or text.startswith("a"): return 1
    if text == "b" or text.startswith("b"): return 0

    return None


def compute_metrics(results):
    total    = sum(r["pred"] is not None for r in results)
    correct  = sum(r["pred"] == r["gt"] for r in results if r["pred"] is not None)
    acc      = correct / total if total > 0 else 0
    tie_rate = sum(r["tie"] for r in results) / len(results)
    return acc, tie_rate


print("Helper functions ready.")

Helper functions ready.


In [ ]:
# CoT evaluation function
# Matches original evaluate_attribute CoT section exactly
# — samples from df_votes (individual votes as ground truth)
# — runs twice with position swap
# — flips pred2 after swap
# — uses step-by-step CoT reasoning prompt
def evaluate_cot(attribute_name, study_id, n_samples=50, eval_model=None):
    if eval_model is None:
        eval_model = inference_model

    # Sample from df_votes — individual votes as ground truth
    # This matches original notebook evaluate_attribute exactly
    votes = df_votes[df_votes["study_id"] == study_id].copy()
    print(f"{attribute_name} — votes after study filter:", len(votes))

    votes = votes[
        votes["left"].isin(image_map.keys()) &
        votes["right"].isin(image_map.keys())
    ].copy()
    print(f"{attribute_name} — votes after image filter:", len(votes))

    records = []
    for _, row in votes.iterrows():
        img_A = image_map[row["left"]]
        img_B = image_map[row["right"]]
        label = 1 if row["choice"] == "left" else 0
        records.append({"img_A": img_A, "img_B": img_B, "label": label})

    votes_df = pd.DataFrame(records)
    if len(votes_df) < n_samples:
        n_samples = len(votes_df)
    eval_df = votes_df.sample(n=n_samples, random_state=42).reset_index(drop=True)

    print("votes_df length:", len(votes_df))
    print("eval_df length:", len(eval_df))

    cues     = ATTRIBUTE_CUES.get(attribute_name.lower(), [])
    cue_text = ", ".join(cues)

    # CoT reasoning prompt — adapted from original notebook
    # Uses [INST]/[/INST] format for LLaVA-NeXT
    # Keeps the step-by-step reasoning structure from original
    cot_prompt = (
        f"[INST] <image>\n\n"
        f"Two places are shown side-by-side:\n\n"
        f"Image A (left)\n"
        f"Image B (right)\n\n"
        f"Your task is to determine which place looks more **{attribute_name}**.\n\n"
        f"Important visual cues related to \"{attribute_name}\" may include:\n"
        f"{cue_text}\n\n"
        f"Follow these steps:\n\n"
        f"Step 1: Briefly describe Image A focusing on relevant cues.\n"
        f"Step 2: Briefly describe Image B focusing on relevant cues.\n"
        f"Step 3: Compare both images based on those cues.\n"
        f"Step 4: Decide which image better represents \"{attribute_name}\".\n\n"
        f"IMPORTANT:\n"
        f"Give your final answer EXACTLY in this format on a new line:\n"
        f"Answer: A\n"
        f"or\n"
        f"Answer: B\n"
        f" [/INST]"
    )

    # Helper to move inputs to GPU with correct dtype
    def to_device(inputs):
        return {
            k: v.to("cuda").half() if hasattr(v, "to") and v.is_floating_point()
            else v.to("cuda") if hasattr(v, "to")
            else v
            for k, v in inputs.items()
        }

    cot_results = []

    for i in tqdm(range(len(eval_df)), desc=f"CoT: {attribute_name}"):
        img_A = eval_df.iloc[i]["img_A"]
        img_B = eval_df.iloc[i]["img_B"]
        gt    = eval_df.iloc[i]["label"]

        # Run normal order
        combined_img = concatenate_images(img_A, img_B)
        inputs = processor(text=cot_prompt, images=combined_img, return_tensors="pt")
        inputs = to_device(inputs)
        output = eval_model.generate(
            **inputs, max_new_tokens=80, temperature=0.0, do_sample=False
        )
        pred1 = parse_prediction(
            processor.decode(output[0], skip_special_tokens=True)
        )

        # Run swapped order
        combined_img_swapped = concatenate_images(img_B, img_A)
        inputs = processor(text=cot_prompt, images=combined_img_swapped, return_tensors="pt")
        inputs = to_device(inputs)
        output = eval_model.generate(
            **inputs, max_new_tokens=80, temperature=0.0, do_sample=False
        )
        pred2 = parse_prediction(
            processor.decode(output[0], skip_special_tokens=True)
        )

        # Flip second prediction — same as original notebook
        if pred2 is not None:
            pred2 = 1 - pred2

        # Tie detection — same as original notebook
        is_tie = (pred1 is not None and pred2 is not None and pred1 != pred2)

        # Final prediction — majority vote
        votes_list = [v for v in [pred1, pred2] if v is not None]
        if len(votes_list) == 0:
            final_pred = None
        elif votes_list.count(1) > votes_list.count(0):
            final_pred = 1
        elif votes_list.count(0) > votes_list.count(1):
            final_pred = 0
        else:
            final_pred = pred1

        cot_results.append({"pred": final_pred, "gt": gt, "tie": is_tie})

    cot_acc, cot_tie = compute_metrics(cot_results)

    print("CoT valid predictions:", sum(r["pred"] is not None for r in cot_results))
    print("Total samples:", len(eval_df))

    return {
        "attribute":    attribute_name,
        "cot_acc":      round(cot_acc, 3),
        "cot_tie_rate": round(cot_tie, 3),
    }


print("evaluate_cot ready.")

evaluate_cot ready.


In [ ]:
# Move model to GPU for evaluation
inference_model = model_module.model
inference_model = inference_model.to("cuda")
inference_model.eval()
processor.tokenizer.padding_side = "left"
print("Model device:", next(inference_model.parameters()).device)

Model device: cuda:0


In [ ]:
# ============================================================
# Quick check on 'safer' first — ~13 minutes
# Baseline (LLaVA-1.5): cot_acc=0.540, cot_tie_rate=0.42
# ============================================================

N_EVAL_SAMPLES = 50

print("Quick CoT evaluation: safer")
print("Baseline (LLaVA-1.5): cot_acc=0.540, cot_tie_rate=0.42\n")

res = evaluate_cot("safer", ATTRIBUTE_MAP["safer"], n_samples=N_EVAL_SAMPLES)

print("\n=== RESULTS ===")
print(res)
print(f"\nBaseline cot_acc:      0.540  →  SFT cot_acc:      {res['cot_acc']}")
print(f"Baseline cot_tie_rate: 0.42   →  SFT cot_tie_rate: {res['cot_tie_rate']}")

Quick CoT evaluation: safer
Baseline (LLaVA-1.5): cot_acc=0.540, cot_tie_rate=0.42

safer — votes after study filter: 443052
safer — votes after image filter: 440758
votes_df length: 440758
eval_df length: 50


CoT: safer:   0%|          | 0/50 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
CoT: safer:   2%|▏         | 1/50 [00:20<16:56, 20.74s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
CoT: safer:   4%|▍         | 2/50 [00:41<16:33, 20.70s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
CoT: safer:   6%|▌         | 3/50 [01:02<16:12, 20.68s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
CoT: safer:   8%|▊         | 4/50 [01:22<15:49, 20.64s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
CoT: safer:  10%|█         | 5/50 [01:

CoT valid predictions: 50
Total samples: 50

=== RESULTS ===
{'attribute': 'safer', 'cot_acc': np.float64(0.52), 'cot_tie_rate': 1.0}

Baseline cot_acc:      0.540  →  SFT cot_acc:      0.52
Baseline cot_tie_rate: 0.42   →  SFT cot_tie_rate: 1.0


In [ ]:
# ============================================================
# Full evaluation across all 6 attributes
# Only run if safer results above look promising
# Takes ~80 minutes for all 6 attributes at 50 samples each
# ============================================================

N_EVAL_SAMPLES = 50

# Baseline from original notebook (LLaVA-1.5, no fine-tuning)
BASELINES = {
    "beautiful":       {"cot_acc": 0.590, "cot_tie_rate": 0.51},
    "safer":           {"cot_acc": 0.540, "cot_tie_rate": 0.42},
    "livelier":        {"cot_acc": 0.590, "cot_tie_rate": 0.69},
    "more boring":     {"cot_acc": 0.510, "cot_tie_rate": 0.61},
    "wealthier":       {"cot_acc": 0.540, "cot_tie_rate": 0.46},
    "more depressing": {"cot_acc": 0.500, "cot_tie_rate": 0.40},
}

attributes = [
    "beautiful",
    "safer",
    "livelier",
    "more boring",
    "wealthier",
    "more depressing",
]

results = []
for attr in attributes:
    print(f"\n==============================")
    print(f"Running evaluation for: {attr}")
    print(f"==============================\n")
    res = evaluate_cot(attr, ATTRIBUTE_MAP[attr], n_samples=N_EVAL_SAMPLES)
    results.append(res)

results_df = pd.DataFrame(results)
results_df["baseline_cot_acc"]      = results_df["attribute"].map(
    lambda x: BASELINES[x]["cot_acc"]
)
results_df["baseline_cot_tie_rate"] = results_df["attribute"].map(
    lambda x: BASELINES[x]["cot_tie_rate"]
)
results_df["acc_change"] = (
    results_df["cot_acc"] - results_df["baseline_cot_acc"]
).round(3)
results_df["tie_change"] = (
    results_df["cot_tie_rate"] - results_df["baseline_cot_tie_rate"]
).round(3)

print("\n=== FULL SFT RESULTS VS BASELINE ===")
print(results_df[[
    "attribute",
    "baseline_cot_acc", "cot_acc", "acc_change",
    "baseline_cot_tie_rate", "cot_tie_rate", "tie_change"
]].to_string(index=False))

results_path = os.path.join(RESULTS_DIR, "sft_cot_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nResults saved to: {results_path}")